# Planner Agent Tests

Use this notebook to validate the Planner agent end to end: raw user question → refined, schema-grounded question.

Before running the notebook, make sure:
- `data/askdata.db` exists
- `GOOGLE_API_KEY` is available in the environment used by the notebook kernel

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path("..").resolve()
dotenv_path = project_root / ".env"

if not project_root.exists():
    raise FileNotFoundError(
        "Could not find the AskData project root from the current notebook working directory."
    )

if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

load_dotenv(dotenv_path=dotenv_path, override=True)

print(f"Project root: {project_root}")
print(f"Loaded environment from: {dotenv_path}")
print(f"GOOGLE_API_KEY loaded: {bool(os.getenv('GOOGLE_API_KEY'))}")

Project root: /Users/zac_burns/Documents/Personal/AskData
Loaded environment from: /Users/zac_burns/Documents/Personal/AskData/.env
GOOGLE_API_KEY loaded: True


In [2]:
from askdata.agents.config import AgentConfig
from askdata.agents.planner import PlannerAgent
from askdata.storage import SQLiteDatabase

In [3]:
database = SQLiteDatabase(project_root / "data" / "askdata.db")
database.ensure_exists()

schema = database.get_schema_summary()

print("Available tables:", database.list_tables())
print()
for line in schema.split("\n"):
    if not line.strip():
        continue
    parts = line.split(": ", 1)
    if len(parts) != 2:
        continue
    table, columns = parts
    print(f"\n{table}")
    print("-" * 40)
    for col in columns.split(", "):
        print(f"  • {col}")

Available tables: ['customers', 'geolocation', 'order_items', 'orders', 'payments', 'products', 'sellers', 'walmart_trustpilot_reviews']


- customers
----------------------------------------
  • customer_id TEXT
  • customer_unique_id TEXT
  • customer_zip_code_prefix INTEGER
  • customer_city TEXT
  • customer_state TEXT

- geolocation
----------------------------------------
  • geolocation_zip_code_prefix INTEGER
  • geolocation_lat REAL
  • geolocation_lng REAL
  • geolocation_city TEXT
  • geolocation_state TEXT

- order_items
----------------------------------------
  • order_id TEXT
  • order_item_id INTEGER
  • product_id TEXT
  • seller_id TEXT
  • shipping_limit_date TEXT
  • price REAL
  • freight_value REAL

- orders
----------------------------------------
  • order_id TEXT
  • customer_id TEXT
  • order_status TEXT
  • order_purchase_timestamp TEXT
  • order_approved_at TEXT
  • order_delivered_carrier_date TEXT
  • order_delivered_customer_date TEXT
  • order_estimated_

In [4]:
agent = PlannerAgent(config=AgentConfig())
agent

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Helper

Prints the original and refined questions side by side.

In [5]:
def plan(question: str) -> str:
    result = agent.run(question, schema_summary=schema)
    sep = "─" * 60
    print(f"{'📥 Original question':^60}")
    print(sep)
    print(result.original_question)
    print()
    print(f"{'✏️  Refined question':^60}")
    print(sep)
    print(result.refined_question)
    return result.refined_question

## Example 1 — Vague question

"Who are the top sellers?" is ambiguous: top by revenue? units sold? in what period?
The planner should ground it in the schema and surface those choices.

In [6]:
plan("Who are the top sellers?")

                    📥 Original question                     
────────────────────────────────────────────────────────────
Who are the top sellers?

                    ✏️  Refined question                    
────────────────────────────────────────────────────────────
Which sellers have the highest total sales revenue, calculated as the sum of the price of all items sold?


'Which sellers have the highest total sales revenue, calculated as the sum of the price of all items sold?'

## Example 2 — Ambiguous time reference

"Show me recent orders" contains an undefined time window.
The planner should either resolve or flag the ambiguity using the schema.

In [7]:
plan("Show me recent orders with a high payment value")

                    📥 Original question                     
────────────────────────────────────────────────────────────
Show me recent orders with a high payment value

                    ✏️  Refined question                    
────────────────────────────────────────────────────────────
What are the order IDs, order purchase timestamps, and payment values for orders with a payment value greater than the average payment value, ordered by the most recent purchase timestamp?


'What are the order IDs, order purchase timestamps, and payment values for orders with a payment value greater than the average payment value, ordered by the most recent purchase timestamp?'